# Example 2: Relativistic Time Conversions in LuPNT

This tutorial covers how LuPNT converts between relativistic time scales, and
how accurate each route is.

## How time conversion works in LuPNT

A time scale is a coordinate on a particular 4-D reference system. Converting
between two of them means evaluating a *relativistic* relationship, not adding a
constant. LuPNT organises this as a **graph**: each time scale is a node, and
each registered edge returns the **offset** between two scales,

$$\delta_{A\to B}(t) \;=\; t_B - t_A ,$$

given the epoch's reading in scale $A$. A conversion finds the shortest
registered route and sums the offsets along it.

| class | edges | cost |
|---|---|---|
| constant | TAI↔TT (32.184 s), TAI↔GPS (19 s) | arithmetic |
| table | TAI↔UTC (leap seconds), UTC↔UT1 (Earth orientation) | lookup |
| linear | TT↔TCG, TDB↔TCB, TCL↔LT | rescale by $L\sim10^{-8}$ |
| model | TT↔TDB | Chebyshev fit / analytic series |
| integral | TDB↔TCL | trapezoidal sweep from $T_0$ (1977) |

Two consequences matter in practice.

**Offsets, not absolute epochs.** An absolute epoch is a `float64` count of
seconds from J2000, so near 2030 one unit in the last place is

$$|t|\,2^{-52} \approx 2.45\times10^{-7}\ \text{s} \approx 245\ \text{ns}
\approx 73\ \text{m}\times c .$$

Summing small offsets never differences two large numbers, so `Epoch` keeps full
double precision. `convert_time` has to *return* an absolute epoch and is
therefore capped near that 245 ns floor no matter how good the model is. Use
`Epoch` or the offset accessors (`tt_minus_tdb`, `tdb_minus_tcl`,
`tdb_minus_lt`) whenever the difference itself is the quantity you care about.

**Routing matters.** TCL↔LT is a pure linear rescaling; reaching it via TAI would
drag in the TDB↔TCL integral, which sweeps from 1977 on every call. The graph
uses the direct edge instead.

This notebook compares the three TT−TDB routes LuPNT offers, then uses the
integral-based TDB−LT relation to reach TT, and finally decomposes the periodic
structure.


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import os, sys
from pathlib import Path

# Use THIS repo's pylupnt/data, not a stale copy on a global PYTHONPATH (~/.zshrc) or kernelspec.
for _b in [Path.cwd(), *Path.cwd().parents]:
    if (_b / "python/pylupnt/__init__.py").exists():
        sys.path.insert(0, str((_b / "python").resolve()))
        if (_b / "data/LuPNT_data/ephemeris").is_dir():
            os.environ["LUPNT_DATA_PATH"] = str((_b / "data/LuPNT_data").resolve())
        break
import pylupnt as pnt

# --- Fundamental constants ---
L_G = pnt.L_G  # TT-TCG scale: d(TCG-TT)/dTT = L_G/(1-L_G)
L_B = pnt.L_B  # TDB-TCB scale
L_L = pnt.L_L  # LT-TCL scale: LT runs slower than TCL on the lunar geoid
L_EM = 1.7093906e-11  # mean LT-TT rate correction (Fienga+2023; not yet in Python API)
C = 299792458  # speed of light [m/s]
SECS_DAY = pnt.SECS_DAY
TDB_0 = -65.5e-6  # [s] TDB offset (DE405 ephemeris, from LuPNT constants.h)

# Reference epoch T_0 = 1977-01-01 00:00:32.184 TAI
T0_MJD = pnt.MJD_COORDINATE_TT_TCG_TCB  # 43144.0003725
T0_TT = pnt.mjd_to_time(T0_MJD)  # [s] from J2000 in TT
MJD_J2000 = pnt.MJD_J2000_TT  # 51544.5

print(f"Reference epoch T_0: MJD {T0_MJD}  =  {T0_TT:.3f} s from J2000 (TT)")
print(f"L_G={L_G:.4e}, L_B={L_B:.4e}, L_L={L_L:.4e}, L_EM={L_EM:.4e}")
print(f"Speed of light: c = {C:.0f} m/s")

## 1. TT − TDB: three routes

TDB is the barycentric coordinate time in which JPL publishes its ephemerides;
TT is the terrestrial scale that clocks on the geoid realise. Their difference is
dominated by an annual term of amplitude $\sim1.66$ ms, driven by the Earth's
eccentric orbit: the Earth's speed and its depth in the Sun's potential both vary
over the year, so a terrestrial clock gains and loses against a barycentric one.

LuPNT offers three routes, in increasing order of fidelity:

1. **DE440 Eq. (3) integral** — the relativistic definition, integrated directly
   (Park et al. 2021). Opt in with `set_tt_tdb_model(DE440_INTEGRAL)`.

$$\mathrm{TDB}-\mathrm{TT} = \frac{L_G-L_B}{1-L_B}(\mathrm{TDB}-T_0)
  + \frac{1-L_G}{1-L_B}\!\int_{T_0}^{\mathrm{TDB}}\!
    \frac{1}{c^{2}}\Big(\tfrac12 v_E^2 + w_{0E}\Big)\,dt \;-\;\dots$$

2. **DE440t kernel** — JPL's own integrated TT−TDB, read from the `de440t.bsp`
   time-ephemeris segment. This is the reference.

3. **Chebyshev fit** of that kernel — what LuPNT uses by default (built on demand
   by `set_tt_tdb_auto_fit`, on by default).


In [ ]:
SECS_DAY = pnt.SECS_DAY
t_start = float(pnt.gregorian_to_time("2020-01-01T00:00:00"))
t_end = float(pnt.gregorian_to_time("2035-01-01T00:00:00"))
n_pts = 721  # ~7.6 day sampling over 15 years
t_tdb = np.linspace(t_start, t_end, n_pts)
years = 2000.0 + t_tdb / SECS_DAY / 365.25

ulp = np.abs(t_tdb).max() * 2.0**-52
print(f"grid: {n_pts} points, {years[0]:.1f} - {years[-1]:.1f}")
print(
    f"absolute-epoch ULP here: {ulp*1e9:.1f} ns  (every route below returns OFFSETS)\n"
)

# --- Route 2: the DE440t kernel (reference) -----------------------------------
tt_tdb_kernel = np.asarray(
    pnt.spice.get_time_ephemeris_offset(t_tdb, 1000000001), float
)  # TT - TDB [s]

# --- Route 1: DE440 Eq. (3) relativistic integral ------------------------------
pnt.set_de440_tt_tdb_step(0.01 * SECS_DAY)  # 864 s trapezoid
_t = time.time()
tt_tdb_integral = -np.asarray(pnt.tdb_minus_tt_de440(t_tdb), float)
t_integral = time.time() - _t

# --- Route 3: Chebyshev fit of the kernel (LuPNT default) ---------------------
pnt.init_tt_minus_tdb_fit(t_start - 30 * SECS_DAY, t_end + 30 * SECS_DAY)
_t = time.time()
tt_tdb_fit = np.asarray(pnt.tt_minus_tdb(t_tdb), float)
t_fit = time.time() - _t

print(f"Eq.(3) integral : {t_integral:7.3f} s for {n_pts} points (single sorted sweep)")
print(f"Chebyshev fit   : {t_fit:7.3f} s")

In [ ]:
SECS_YEAR = 365.25 * SECS_DAY


def decompose(resid, t):
    """Split a residual into constant + linear drift + what is left."""
    A = np.vstack([np.ones_like(t), (t - t[0]) / SECS_YEAR]).T
    c, *_ = np.linalg.lstsq(A, resid, rcond=None)
    return c[0], c[1], resid - A @ c


print(f"{'route':34s} {'rms':>12s} {'drift':>14s} {'detrended rms':>15s}")
print("-" * 78)
for name, v in [
    ("DE440 Eq.(3) integral", tt_tdb_integral),
    ("Chebyshev fit of the kernel", tt_tdb_fit),
]:
    r = (v - tt_tdb_kernel) * 1e9  # [ns]
    const, drift, detr = decompose(r, t_tdb)
    print(
        f"{name:34s} {np.sqrt((r**2).mean()):9.4f} ns {drift:+11.4f} ns/yr "
        f"{np.sqrt((detr**2).mean()):12.4f} ns"
    )

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), constrained_layout=True, sharex=True)

axes[0].plot(years, tt_tdb_kernel * 1e3, lw=1.0, color="k")
axes[0].set_ylabel("TT - TDB  [ms]")
axes[0].set_title("TT - TDB from the DE440t kernel: the 1.66 ms annual term")
axes[0].grid(True, lw=0.4, alpha=0.5)

r_int = (tt_tdb_integral - tt_tdb_kernel) * 1e9
const, drift, detr_int = decompose(r_int, t_tdb)
axes[1].plot(years, r_int, lw=1.0, color="tab:red", label="Eq.(3) - kernel")
axes[1].plot(
    years,
    r_int - detr_int,
    lw=1.0,
    ls="--",
    color="tab:orange",
    label=f"fitted drift {drift:+.3f} ns/yr",
)
axes[1].set_ylabel("[ns]")
axes[1].set_title("Route 1 vs reference: an almost purely secular difference")
axes[1].legend(fontsize=9)
axes[1].grid(True, lw=0.4, alpha=0.5)

axes[2].plot(years, (tt_tdb_fit - tt_tdb_kernel) * 1e12, lw=1.0, color="tab:green")
axes[2].set_ylabel("[ps]")
axes[2].set_xlabel("year")
axes[2].set_title("Route 3 vs reference: the fit reproduces the kernel to picoseconds")
axes[2].grid(True, lw=0.4, alpha=0.5)
plt.show()

**Reading the numbers.**

The Chebyshev fit reproduces the kernel to sub-picosecond — it interpolates the
same data, and the training nodes are read as clean offsets, so nothing limits it
but the polynomial degree. This is why it is the default.

The Eq. (3) integral is a genuinely independent computation: it integrates the
relativistic definition from LuPNT's own ephemeris rather than reading JPL's
answer. Its difference from the kernel is *almost purely secular* — after
removing a constant and a linear term only a few picoseconds remain, so the
periodic physics matches and only the rate differs slightly.

That residual rate comes from the small bodies. DE440 integrates 343 asteroids,
30 KBOs and a Kuiper ring at 44 au; LuPNT has no ephemerides for them, so
$w_{0E}$ adds ring models of the two populations using published masses
(`GM_ASTEROID_BELT`, `GM_KUIPER_BELT`), which account for most but not all of the
difference.


## 2. TDB − LT by integration, then on to TT

LT (Lunar Time) is the analogue of TT for a clock on the lunar selenoid. Reaching
it from TDB uses the one genuinely expensive edge in the graph — the TDB↔TCL
integral, which sweeps from $T_0$ = 1977 — followed by the cheap linear TCL↔LT
rescaling by $L_L$.

The chain to TT is then

$$\mathrm{LT}-\mathrm{TT} \;=\; \underbrace{(\mathrm{LT}-\mathrm{TDB})}_{\text{integral}}
  \;+\; \underbrace{(\mathrm{TDB}-\mathrm{TT})}_{\text{Part 1}} ,$$

which is exactly what `Epoch` composes when asked for `LT -> TT`. Every term is a
small offset, so the ~245 ns epoch floor never enters.


In [ ]:
# TDB - LT via the integral-based route.
_t = time.time()
tdb_lt = np.asarray(pnt.tdb_minus_lt(t_tdb), float)  # TDB - LT [s]
print(f"TDB - LT: {time.time()-_t:.2f} s for {n_pts} points")

# Chain to TT.  LT - TT = (LT - TDB) + (TDB - TT)
lt_tt = -tdb_lt - tt_tdb_fit  # [s]

# Cross-check the chain against Epoch, which walks the graph itself.
# time_scale_offset(Epoch(LT), TT) returns TT - LT, so negate it to compare
# against lt_tt = LT - TT.
_chk = slice(0, n_pts, max(1, n_pts // 40))
lt_tt_epoch = -np.array(
    [
        float(
            pnt.time_scale_offset(
                pnt.Epoch.from_seconds(float(x), pnt.Time.LT), pnt.Time.TT
            )
        )
        for x in (t_tdb[_chk] - tdb_lt[_chk])  # LT reading of the same instant
    ]
)
print(
    f"chain vs Epoch(LT->TT): max |diff| = "
    f"{np.abs(lt_tt[_chk] - lt_tt_epoch).max()*1e9:.4f} ns"
)

days = (t_tdb - t_tdb[0]) / SECS_DAY
slope_us_per_day = np.polyfit(days, lt_tt * 1e6, 1)[0]
print(f"\nsecular drift of LT - TT: {slope_us_per_day:.4f} us/day")
print("  (Turyshev et al. 2025 give 56.0256 us/day)")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), constrained_layout=True, sharex=True)

axes[0].plot(years, lt_tt / 86400.0 * 1e3, lw=1.0, color="tab:purple")
axes[0].set_ylabel("LT - TT  [ms]")
axes[0].set_title("LT - TT: a lunar clock runs fast by ~56 us/day")
axes[0].grid(True, lw=0.4, alpha=0.5)

lt_tt_periodic = lt_tt - np.polyval(np.polyfit(days, lt_tt, 1), days)
axes[1].plot(years, lt_tt_periodic * 1e6, lw=0.9, color="tab:purple")
axes[1].set_ylabel("[us]")
axes[1].set_xlabel("year")
axes[1].set_title("Periodic part, secular trend removed")
axes[1].grid(True, lw=0.4, alpha=0.5)
plt.show()

## 3. Fourier decomposition

With the secular trend removed, what is left is a sum of periodic terms whose
frequencies are the orbital periods of the system. Decomposing them identifies
which physical motion drives each contribution.

Expected lines:

* **TT − TDB**: the Earth's orbit — 1 year (the 1.66 ms elliptical term), plus
  smaller planetary terms (Jupiter ~11.9 yr, Saturn ~29.5 yr).
* **LT − TT**: the Moon's orbit — the anomalistic month (27.55 d), the synodic
  month (29.53 d), and half-month harmonics.


## 3. Fourier decomposition

With the secular trend removed, what remains is a sum of periodic terms whose
frequencies are the orbital periods driving each effect.

**Sampling governs what can be seen.** The 15-year grid above has ~7.6 day
spacing, so its Nyquist period is ~15 days. The lunar terms sit at 27.55 d
(anomalistic month) and 29.53 d (synodic month) — under 4 samples per period,
which aliases them and smears power into neighbouring bins. The Earth's annual
term is sampled ~48 times per period and is fine.

So the two signals need different grids: the 15-year record resolves the annual
and planetary structure, while a shorter, densely-sampled record is required for
the lunar lines. That is a general lesson, not an artefact of this notebook —
resolution is set by record *length*, aliasing by sample *spacing*.


In [ ]:
def spectrum(sig, t):
    """Amplitude spectrum of a detrended, uniformly-sampled signal."""
    n = len(sig)
    dt = t[1] - t[0]
    x = sig - sig.mean()
    w = np.hanning(n)
    amp = np.abs(np.fft.rfft(x * w)) / (w.sum() / 2.0)
    freq = np.fft.rfftfreq(n, d=dt)
    with np.errstate(divide="ignore"):
        period_d = 1.0 / (freq * SECS_DAY)
    return period_d[1:], amp[1:], freq[1:]


def peaks(period_d, amp, freq, pmin, pmax, n=4, sep_bins=3.0):
    """Strongest lines in a period band, one per resolution element.

    Spectral leakage puts sidelobes about ONE frequency bin either side of a
    real line, so the separation floor has to be expressed in bins -- a fixed
    log-period spacing is too wide at short periods and too narrow at long ones.
    A Hanning mainlobe spans ~4 bins; 3 is enough to reject its skirts while
    keeping genuinely distinct lines.
    """
    df = freq[1] - freq[0]
    m = (period_d >= pmin) & (period_d <= pmax)
    p, a, f = period_d[m], amp[m], freq[m]
    out = []
    for i in np.argsort(a)[::-1]:
        if all(abs(f[i] - fq) > sep_bins * df for _, _, fq in out):
            out.append((p[i], a[i], f[i]))
        if len(out) == n:
            break
    return [(q, b) for q, b, _ in out]


# --- TT - TDB: the 15-year record resolves the annual and planetary terms ----
_, _, tt_tdb_periodic = decompose(tt_tdb_kernel, t_tdb)
p1, a1, f1 = spectrum(tt_tdb_periodic, t_tdb)
print(
    f"TT - TDB   ({n_pts} pts over 15 yr, {(t_tdb[1]-t_tdb[0])/SECS_DAY:.1f} d spacing)"
)
# Cap the band at ~1/3 of the record: longer "periods" are the residual trend
# leaking, not planetary terms.
for p, a in peaks(p1, a1, f1, pmin=100, pmax=5.0 * 365.25, n=4):
    print(f"   {p/365.25:8.3f} yr   amplitude {a*1e6:9.2f} us")

# --- LT - TT: the record must be long enough to SEPARATE the lunar months ----
# They sit 2.4e-3 /d apart in frequency, so a record of T days resolves them by
# 2.4e-3*T bins. A Hanning mainlobe spans ~4 bins, so T must exceed ~4.5 years;
# 3 years is not enough and merges them however the peaks are picked.
n_years = 6.0
n_lun = int(n_years * 365.25 * 4) + 1  # 0.25 day spacing
t_lun = np.linspace(t_start, t_start + n_years * 365.25 * SECS_DAY, n_lun)
pnt.init_lt_minus_tt_fit(t_lun[0] - 30 * SECS_DAY, t_lun[-1] + 30 * SECS_DAY)
lt_tt_lun = -np.asarray(pnt.tdb_minus_lt(t_lun), float) - np.asarray(
    pnt.tt_minus_tdb(t_lun), float
)
d_lun = (t_lun - t_lun[0]) / SECS_DAY
lt_tt_lun_per = lt_tt_lun - np.polyval(np.polyfit(d_lun, lt_tt_lun, 1), d_lun)
p2, a2, f2 = spectrum(lt_tt_lun_per, t_lun)
_sep = abs(1 / 29.5306 - 1 / 27.5545) * n_years * 365.25
print(
    f"\nLT - TT    ({n_lun} pts over {n_years:.0f} yr, 0.25 d spacing;"
    f" the two lunar months are {_sep:.1f} bins apart)"
)
for p, a in peaks(p2, a2, f2, pmin=5, pmax=200, n=5):
    print(f"   {p:8.3f} d    amplitude {a*1e6:9.4f} us")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)

axes[0].loglog(p1 / 365.25, a1 * 1e6, lw=0.9, color="tab:blue")
axes[0].axvline(1.0, color="k", ls=":", lw=0.8)
axes[0].text(1.0, a1.max() * 1e6 * 0.5, "1 yr", rotation=90, fontsize=8, ha="right")
axes[0].set_xlabel("period [yr]")
axes[0].set_ylabel("amplitude [us]")
axes[0].set_title("TT - TDB  (15 yr record)")
axes[0].grid(True, which="both", lw=0.3, alpha=0.5)

axes[1].loglog(p2, a2 * 1e6, lw=0.9, color="tab:purple")
for d, lab in [(27.5545, "anomalistic"), (29.5306, "synodic"), (13.6061, "half-anom.")]:
    axes[1].axvline(d, color="k", ls=":", lw=0.8)
    axes[1].text(d, a2.max() * 1e6 * 0.4, lab, rotation=90, fontsize=8, ha="right")
axes[1].set_xlim(5, 200)
axes[1].set_xlabel("period [d]")
axes[1].set_ylabel("amplitude [us]")
axes[1].set_title("LT - TT  (3 yr record, 0.25 d sampling)")
axes[1].grid(True, which="both", lw=0.3, alpha=0.5)
plt.show()